<a href="https://colab.research.google.com/github/anthonyho03-coder/Special-Topics---MIS-Agentic-A.I.-Anthony-Ho/blob/main/Exercise2_Code_Generation_ReACT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2 — Code Generation with ReACT Prompting

**Tool used:** Prompt tested in ChatGPT / OpenAI Playground (reasoning transcript below is the captured response); code cells here in Colab are the actual executed implementation and test evidence.

**Goal:** Use a ReACT (Reason -> Act -> Observe) style prompt to generate Python code, then actually execute it and verify correctness against edge cases.

## Full ReACT Prompt

**System prompt:**
```
You are a Python coding assistant that uses a Reason-then-Act approach. Before writing
any code, think through the requirements and edge cases. Only after reasoning should you
produce the code. After producing code, verify it against edge cases and fix any issues.
```

**User prompt:**
```
Task: Write a Python function filter_primes(nums) that returns only the prime numbers
from a list, using a helper function is_prime(n).

Follow this exact process:
1. REASON: Explain what defines a prime number and list the edge cases to handle
   (negative numbers, 0, 1, duplicates).
2. ACT: Write is_prime(n) and filter_primes(nums) based on that reasoning.
3. OBSERVE: Test the function against the edge cases and report whether it passes.
   If it fails, fix the code and re-test.
```

## Stage 1 — REASON (captured model output)

> A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself.
>
> Edge cases to handle:
> - Numbers <= 1 (0, 1, and negatives) are **not** prime and must return `False`.
> - Negative numbers must not crash `is_prime` (a naive `sqrt()`-based check can error on negative input).
> - Duplicates in the input list should be **preserved**, not deduplicated, since the task doesn't ask for uniqueness.
>
> **Approach:** implement `is_prime(n)` using trial division up to `sqrt(n)` for efficiency, guarding against `n < 2` up front. Then `filter_primes(nums)` applies `is_prime` via a list comprehension.

## Stage 2 — ACT (first attempt, intentionally kept close to the reasoning but with one gap, to show the Observe/fix cycle)

In [ ]:
import math

# --- First attempt ---
def is_prime_v1(n):
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def filter_primes_v1(nums):
    return [n for n in nums if is_prime_v1(n)]

print("v1 defined.")


v1 defined.


## Stage 3 — OBSERVE (test the first attempt against edge cases)

In [ ]:
test_cases = [-5, 0, 1, 2, 3, 4, 17, 18, 19, 20, 2, 3]  # includes negative, 0, 1, and duplicates

try:
    result_v1 = filter_primes_v1(test_cases)
    print("v1 result:", result_v1)
except Exception as e:
    print("v1 FAILED with error:", repr(e))


v1 FAILED with error: ValueError('math domain error')


**Observation:** `filter_primes_v1` crashes on the negative test case. `math.sqrt(-5)` raises a `ValueError` because there's no `n < 2` guard before the square root is taken — this is exactly the edge case flagged in the REASON stage but not yet implemented. This is the "iteration" cycle: reasoning identified the edge case, the first Act missed it, Observe caught it, so we fix and re-test.

## Stage 2 (revised) — ACT (fixed version, addressing the Observe finding)

In [ ]:
# --- Fixed version: adds the n < 2 guard identified during REASON/OBSERVE ---
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def filter_primes(nums):
    return [n for n in nums if is_prime(n)]

print("Fixed is_prime / filter_primes defined.")


Fixed is_prime / filter_primes defined.


## Stage 3 (revised) — OBSERVE (re-test against the same edge cases)

In [ ]:
result = filter_primes(test_cases)
print("Input:  ", test_cases)
print("Output: ", result)

# Explicit edge-case checks
checks = {
    "negative numbers excluded": is_prime(-5) == False,
    "zero excluded": is_prime(0) == False,
    "one excluded": is_prime(1) == False,
    "known prime (17) included": is_prime(17) == True,
    "known composite (18) excluded": is_prime(18) == False,
    "duplicates preserved": test_cases.count(2) == result.count(2) == 2,
}

print("\nEdge-case verification:")
for check, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

assert all(checks.values()), "One or more edge cases failed!"
print("\nAll edge cases passed. filter_primes(nums) is working correctly.")


Input:   [-5, 0, 1, 2, 3, 4, 17, 18, 19, 20, 2, 3]
Output:  [2, 3, 17, 19, 2, 3]

Edge-case verification:
  [PASS] negative numbers excluded
  [PASS] zero excluded
  [PASS] one excluded
  [PASS] known prime (17) included
  [PASS] known composite (18) excluded
  [PASS] duplicates preserved

All edge cases passed. filter_primes(nums) is working correctly.


## Result

The fixed `filter_primes` / `is_prime` pair correctly excludes negative numbers, 0, and 1; correctly includes known primes; correctly excludes known composites; and preserves duplicates. The bug caught during the Observe stage (unhandled negative input causing a `ValueError`) was fixed by adding the `n < 2` guard identified in the original Reason stage.